# Bài 08 · Quá khớp & chính quy hóa: mạng MNIST học thuộc 1.000 ảnh, và những cách chống lại

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/08-chong-qua-khop.ipynb)

Notebook đi kèm [Bài 08 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chong-qua-khop/) (demo tương tác nằm ở trang bài học).
Ta lấy mạng 784 → 128 → 64 → 10 của khóa học nhưng chỉ cho nó học **1.000 ảnh** MNIST, ít đến mức nó học thuộc lòng
được. Ta phát hiện quá khớp bằng tập kiểm định, rồi tự viết bằng numpy các cách chống lại: phạt L2 (weight decay),
dropout, tăng cường dữ liệu và dừng sớm. Cuối cùng ta xem vì sao “mô hình càng lớn càng quá khớp” không phải lúc nào
cũng đúng (double descent).

**Bạn sẽ làm:**
1. Huấn luyện mạng trên 1.000 ảnh trong 150 lượt (epoch), thấy mất mát kiểm định đi lên trong khi mất mát huấn luyện
   về gần 0, và dừng sớm.
2. Xáo ngẫu nhiên các nhãn và xem mạng thuộc lòng cả những cặp (ảnh, nhãn) vô nghĩa.
3. Viết phạt L2 và dropout, đo xem mỗi cách làm gì với trọng số và với đường cong kiểm định.
4. Dịch ảnh ngẫu nhiên, rồi đọc kết quả quét 30 tổ hợp × 3 seed: chọn trên tập kiểm định, báo cáo trên tập kiểm tra.
5. Quét kích thước một mô hình đặc trưng ngẫu nhiên qua mức “vừa đủ để nhớ” và thấy lỗi kiểm định giảm lần thứ hai.

**Cần biết trước:** [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/)
(mini-batch SGD, lượt) và [Bài 07 · Bộ tối ưu](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/) (weight
decay trong AdamW). Mạng, softmax + cross-entropy và lan truyền ngược là của
[Bài 04](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/) và
[Bài 06](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng kết quả quét đầy đủ
(91 lần huấn luyện 150 lượt) được nạp từ đúng tệp mà demo dùng, vì chạy lại mất vài phút; notebook chạy lại bốn lần
huấn luyện trong đó và kiểm tra rằng ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = (
        f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    )  # không in “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 10.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình sang kiểu Việt Nam. Trục log lấy vạch 1–2–5, hoặc lũy thừa của 10 nếu trục rất dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                axis.set_major_locator(
                    FixedLocator([m * 10.0**k for k in range(-6, 6) for m in mult if lo <= m * 10.0**k <= hi])
                )
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527))

## 2 · Dữ liệu: chỉ 1.000 ảnh để học

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28.
Ô dưới tải 4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Ô cũng nạp tệp kết quả quét mà
demo dùng (`fetch_json`, có kiểm tra MD5).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
runs = fetch_json("mnist-overfit.json", "95bbd8b2d0b3e5b27a1cdedbb6e8fd04")  # kết quả quét của demo
print("huấn luyện:", train_img.shape, " kiểm tra:", test_img.shape)

Ta chia dữ liệu như demo, ba phần, mỗi phần một việc:
- **huấn luyện:** 1.000 ảnh, đúng 100 ảnh mỗi chữ số, chọn ngẫu nhiên (seed 0) trong 50.000 ảnh đầu của tệp huấn
  luyện. Đây là “đề cũ” mà mạng được ôn;
- **kiểm định:** 10.000 ảnh cuối của tệp huấn luyện, “đề mới” để phát hiện quá khớp và *chọn* cấu hình;
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, “đề thi thật”, chỉ đọc ở cấu hình đã chọn, không dùng để chọn gì.

In [ ]:
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
pick = np.random.default_rng(0)  # cùng seed với demo, nên cùng 1.000 ảnh
SUB = np.sort(np.concatenate([pick.choice(np.flatnonzero(y_all[:50000] == d), 100, replace=False) for d in range(10)]))
Xtr, ytr = X_all[SUB], y_all[SUB]  # huấn luyện
Xval, yval = X_all[50000:], y_all[50000:]  # kiểm định: phát hiện quá khớp, chọn cấu hình
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # kiểm tra: chỉ để báo cáo
SHUFFLED = pick.permutation(ytr)  # cùng 1.000 nhãn đó, chia lại ngẫu nhiên cho các ảnh (mục 5)
counts = np.bincount(ytr)
print(f"huấn luyện {vn(len(ytr), 0)} ảnh ({counts.min()} ảnh mỗi chữ số, lấy trong {vn(50000, 0)} ảnh đầu)", end=" · ")
print(f"kiểm định {vn(len(yval), 0)} · kiểm tra {vn(len(yte), 0)} ảnh")
print(f"tệp quét của demo: {len(runs['configs'])} cấu hình × {len(runs['seeds'])} seed, mỗi lần {runs['epochs']} lượt")
assert (len(ytr), len(yval), len(yte)) == (runs["nTrain"], runs["nVal"], runs["nTest"]) and (counts == 100).all()

Hình dưới là 2 ảnh huấn luyện đầu tiên của mỗi chữ số, mỗi ảnh ghi nhãn của nó ở trên.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for d in range(10):
    for row, i in enumerate(np.flatnonzero(ytr == d)[:2]):
        axes[row, d].imshow(Xtr[i].reshape(28, 28), cmap="gray_r")
        axes[row, d].set_title(str(d), fontsize=11)
        axes[row, d].axis("off")
plt.show()

## 3 · Mạng của khóa học, và một vòng huấn luyện có ba nút vặn

Mạng của Bài 04–07: $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$, $z = h_2W_3 + b_3$,
softmax + cross-entropy, khởi tạo He, SGD thuần với $\eta = 0{,}1$, batch 64. Với 1.000 ảnh, mỗi lượt chỉ có 16 bước.

Hàm `step()` dưới đây là một bước SGD với lan truyền ngược của Bài 06, cộng thêm hai nút vặn mà các mục sau sẽ giải
thích: `lam` (phạt L2, mục 6) và `p` (dropout, mục 7). Hàm `evaluate()` đo mạng ở **chế độ suy luận**: đủ cả mạng,
không dropout, không cộng phạt. Nó tính bằng float32 cho nhanh gấp đôi; vẫn đủ chính xác cho 4 chữ số.
Hai hàm `dropout_mask()` và `shift_images()` mà `step()` và `train()` gọi tới sẽ được viết ở mục 7 và 8; trước đó ta
chỉ chạy với `p = 0` và không dịch ảnh, nên chúng chưa được gọi.

In [ ]:
SIZES, ETA, BATCH, EPOCHS, PATIENCE, SHIFT = [784, 128, 64, 10], 0.1, 64, 150, 10, 2
Xtr32, Xval32, Xte32 = (X.astype(np.float32) for X in (Xtr, Xval, Xte))


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def logits(params, X):
    """Đi xuôi ở chế độ suy luận: đủ cả mạng, không dropout."""
    z = X
    for k, (W, b) in enumerate(params):
        z = z @ W.astype(np.float32) + b.astype(np.float32)
        z = np.maximum(0, z) if k < 2 else z
    return z


def evaluate(params, X, y):
    """Mất mát cross-entropy trung bình và số ảnh đoán đúng, ở chế độ suy luận."""
    z = logits(params, X)
    m = z.max(axis=1, keepdims=True)
    log_sum = m[:, 0] + np.log(np.exp(z - m).sum(axis=1))  # log Σ e^z, ổn định số (Bài 04)
    return float((log_sum - z[np.arange(len(y)), y]).mean()), int((z.argmax(1) == y).sum())


def step(params, x, y, lam, p, noise):
    """Một bước SGD trên batch (x, y): xuôi (có dropout nếu p > 0), ngược (Bài 06), cập nhật (có phạt L2 nếu lam > 0)."""
    (W1, b1), (W2, b2), (W3, b3) = params
    a1 = x @ W1 + b1
    m1 = (a1 > 0) * (dropout_mask(a1.shape, p, noise) if p else 1)  # đạo hàm ReLU × mặt nạ dropout (mục 7)
    h1 = a1 * m1
    a2 = h1 @ W2 + b2
    m2 = (a2 > 0) * (dropout_mask(a2.shape, p, noise) if p else 1)
    h2 = a2 * m2
    dz = softmax(h2 @ W3 + b3)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = ŷ − y (Bài 04)
    dz /= len(y)
    da2 = (dz @ W3.T) * m2
    da1 = (da2 @ W2.T) * m1
    grads = [(x.T @ da1, da1.sum(0)), (h1.T @ da2, da2.sum(0)), (h2.T @ dz, dz.sum(0))]
    for layer, (gW, gb) in zip(params, grads):
        layer[0] -= ETA * (gW + 2 * lam * layer[0])  # ∂(λ‖W‖²)/∂W = 2λW (mục 6); không phạt độ lệch b
        layer[1] -= ETA * gb

Hàm `train()` chạy 150 lượt và ghi lại, sau mỗi lượt, mất mát và số ảnh đúng trên tập huấn luyện cùng độ lớn trọng số
$\lVert W\rVert^2$ (tổng bình phương mọi phần tử của $W_1, W_2, W_3$). Tập kiểm định tốn thời gian hơn (10.000 ảnh),
nên nó được đo sau mỗi lượt cho đến lượt `dense`, rồi mỗi `val_every` lượt (mặc định 5) và ở lượt cuối. Nút vặn thứ ba, `aug`, dịch ảnh ngẫu nhiên (mục 8).

Hàm cũng **dừng sớm** như demo: giữ lại trọng số có mất mát kiểm định thấp nhất, và ghi lượt dừng khi đã 10 lượt liền
không có kỷ lục mới. Nó vẫn chạy tiếp đến hết để ta thấy chuyện gì xảy ra nếu không dừng.

Như `reel/mnist_mlp/overfit.py`, script tạo dữ liệu cho demo: mỗi seed cho cùng trọng số đầu và cùng thứ tự batch ở
mọi cấu hình, nên khác biệt giữa hai cấu hình chỉ đến từ nút vặn.

In [ ]:
def train(lam=0.0, p=0.0, aug=False, labels=None, seed=0, epochs=EPOCHS, dense=0, val_every=5, stop_early=False):
    labels = ytr if labels is None else labels
    rng = np.random.default_rng(seed)  # trọng số đầu và thứ tự batch
    noise = np.random.default_rng([seed, 1])  # mặt nạ dropout và độ dịch ảnh
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]  # He
    log = {"train": [], "w2": [], "val": {}, "best": None, "stop": None}
    for ep in range(epochs + 1):
        if ep:  # một lượt: 16 batch theo thứ tự ngẫu nhiên
            order = rng.permutation(len(Xtr))
            for i in range(0, len(order), BATCH):
                idx = order[i : i + BATCH]
                x = shift_images(Xtr[idx], noise) if aug else Xtr[idx]  # tăng cường dữ liệu (mục 8)
                step(params, x, labels[idx], lam, p, noise)
        log["train"].append(evaluate(params, Xtr32, labels))
        log["w2"].append(float(sum((W**2).sum() for W, _ in params)))
        if ep <= dense or ep % val_every == 0 or ep == epochs:
            log["val"][ep] = evaluate(params, Xval32, yval)
        if ep <= dense and log["stop"] is None:  # dừng sớm, chờ PATIENCE lượt
            if log["best"] is None or log["val"][ep][0] < log["val"][log["best"]][0]:
                log["best"], log["best_params"] = ep, [[W.copy(), b.copy()] for W, b in params]
            elif ep - log["best"] >= PATIENCE:
                log["stop"] = ep
                if stop_early:
                    break
    log["params"] = params
    return log


def first_perfect(log, n=1000):
    """Lượt đầu tiên mạng đoán đúng cả n ảnh huấn luyện (None nếu chưa bao giờ)."""
    return next((e for e, (_, ok) in enumerate(log["train"]) if ok == n), None)


def check(log, ref):
    """So với đường cong seed 0 mà demo lưu (mất mát làm tròn 4 chữ số, ‖W‖² 1 chữ số; số ảnh đúng lệch ≤ 3)."""
    ep, n = list(log["val"]), len(log["train"])
    assert np.allclose([log["val"][e][0] for e in ep], [ref["valLoss"][e] for e in ep], atol=5e-4)
    assert max(abs(log["val"][e][1] - ref["valOk"][e]) for e in ep) <= 3
    assert np.allclose([v for v, _ in log["train"]], ref["trainLoss"][:n], atol=5e-4)
    assert max(abs(ok - r) for (_, ok), r in zip(log["train"], ref["trainOk"])) <= 3
    assert np.allclose(log["w2"], ref["w2"][:n], atol=0.1), "khác với demo: hãy kiểm tra lại dữ liệu"


def config(lam=0, p=0, aug=False):
    """Một ô trong bảng quét của demo."""
    return next(c for c in runs["configs"] if c["lam"] == lam and c["p"] == p and c["aug"] == aug)


def seed_stats(c, key):
    """Trung bình, min và max qua 3 seed của demo (key: "valOk" hoặc "testOk", ở trọng số dừng sớm)."""
    v = [s[key] / 10000 for s in c["seeds"]]
    return np.mean(v), min(v), max(v)


def fmt_range(c, key):
    m, lo, hi = seed_stats(c, key)
    return f"{pct(m)} ({pct(lo)}–{pct(hi)})"


print(f"{vn(sum(a * b + b for a, b in itertools.pairwise(SIZES)), 0)} tham số cho {vn(len(Xtr), 0)} ảnh huấn luyện")
print(f"{-(-len(Xtr) // BATCH)} bước mỗi lượt, {EPOCHS} lượt = {vn(EPOCHS * -(-len(Xtr) // BATCH), 0)} bước")

## 4 · Học thuộc 1.000 ảnh: điểm cao đề cũ, điểm thấp đề mới

Mạng có 109.386 tham số mà chỉ có 1.000 ảnh để học: hơn 100 tham số cho mỗi ảnh. Ta huấn luyện nó 150 lượt không có
chính quy hóa nào, đo tập kiểm định sau mỗi lượt trong 30 lượt đầu, và so với đường cong seed 0 mà demo lưu.

In [ ]:
base = train(dense=30)
check(base, config())
b_ep = base["best"]
print(f"lần đầu đúng cả 1.000 ảnh huấn luyện: lượt {first_perfect(base)}")
for ep in [0, 10, b_ep, 40, 100, 150]:
    (tl, tok), (vl, vok) = base["train"][ep], base["val"][ep]
    print(
        f"lượt {ep:>3}: huấn luyện mất mát {vn(tl, 4)}, đúng {pct(tok / 1000, 1):>6}  ·  kiểm định mất mát {vn(vl, 4)}, đúng {pct(vok / 10000)}"
    )
rise = base["val"][150][0] / base["val"][b_ep][0] - 1
print(f"mất mát kiểm định thấp nhất ở lượt {b_ep}; đến lượt 150 nó tăng {pct(rise, 0)}")
assert (first_perfect(base), b_ep, base["stop"]) == (26, 20, 30), "khác với demo: hãy kiểm tra lại dữ liệu"

Hình dưới vẽ hai đường theo lượt: mất mát (trái) và tỉ lệ đúng (phải). Huấn luyện là đường xanh liền, chấm
tròn; kiểm định là đường cam nét gạch, chấm vuông. Vạch xanh lá đứng là lượt có mất mát kiểm định thấp nhất.

In [ ]:
TRAIN_STYLE = {"color": BLUE, "marker": "o", "ls": "-", "ms": 3, "lw": 1.6}
VAL_STYLE = {"color": ORANGE, "marker": "s", "ls": (0, (6, 3)), "ms": 3, "lw": 1.6}


def val_curve(log, k=0):
    ep = sorted(log["val"])
    return ep, [log["val"][e][k] for e in ep]


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4))
ep_all = np.arange(EPOCHS + 1)
a.plot(ep_all, [v for v, _ in base["train"]], markevery=10, label="huấn luyện (đề cũ)", **TRAIN_STYLE)
a.plot(*val_curve(base), markevery=5, label="kiểm định (đề mới)", **VAL_STYLE)
a.set(ylim=(0, 1), xlabel="lượt (epoch)", ylabel="mất mát", title="Mất mát (lượt 0 ≈ 2,4, ngoài khung)")
b.plot(ep_all, [ok / 1000 for _, ok in base["train"]], markevery=10, **TRAIN_STYLE)
ep_v, ok_v = val_curve(base, 1)
b.plot(ep_v, np.array(ok_v) / 10000, markevery=5, **VAL_STYLE)
b.set(ylim=(0.8, 1.01), xlabel="lượt (epoch)", ylabel="tỉ lệ đúng", title="Độ chính xác")
for ax in (a, b):
    ax.axvline(b_ep, color=GREEN, lw=1.5)
    ax.grid(alpha=0.3, which="both")
a.annotate(f"thấp nhất: lượt {b_ep}", (b_ep, 0.9), xytext=(6, 0), textcoords="offset points", fontsize=9)
a.legend(frameon=False, fontsize=9, loc="upper right")
vn_axes(fig)
b.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.tight_layout()
plt.show()

Đến lượt 26 mạng đã đúng cả 1.000 ảnh huấn luyện, và mất mát huấn luyện vẫn tiếp tục giảm, về 0,0021 ở lượt 150. Mất mát kiểm định thì thấp nhất ở lượt 20 rồi đi lên: mạng giỏi “đề cũ” hơn mỗi lượt, còn
“đề mới” thì kém đi. Đó là **quá khớp** (overfitting): khoảng cách giữa hai đường cong ngày càng rộng.

### 🤔 Dự đoán trước
Từ lượt 20 đến lượt 150, mất mát kiểm định tăng 30%. Vậy độ chính xác kiểm định ở lượt 150 thấp hơn, bằng, hay cao
hơn ở lượt 20?

In [ ]:
for name, params in [(f"lượt {b_ep}", base["best_params"]), ("lượt 150", base["params"])]:
    p_val = softmax(logits(params, Xval32))
    wrong = p_val.argmax(1) != yval
    loss_i = -np.log(p_val[np.arange(len(yval)), yval])  # mất mát của từng ảnh kiểm định
    print(
        f"{name:>8}: đúng {pct(1 - wrong.mean())} · mất mát trung bình trên ảnh sai {vn(loss_i[wrong].mean(), 2)}, "
        f"trên ảnh đúng {vn(loss_i[~wrong].mean(), 4)} · ảnh sai mà tự tin ≥ 99%: {wrong[p_val.max(1) >= 0.99].sum()}"
    )
assert base["val"][150][1] > base["val"][b_ep][1], "khác với demo: hãy kiểm tra lại dữ liệu"

Độ chính xác kiểm định gần như không đổi, thậm chí nhích lên một chút. Mạng không sai *nhiều ảnh hơn*; nó sai *tự tin
hơn*. Mất mát trung bình trên những ảnh nó đoán sai tăng từ 2,85 lên 4,41, và số ảnh sai mà mạng tự tin ≥ 99% tăng từ
21 lên 132. Cross-entropy phạt rất nặng một dự đoán sai mà tự tin (Bài 04), nên chỉ mất mát mới thấy điều này. Quá khớp không nhất thiết
hiện ra ở độ chính xác; hãy theo dõi cả mất mát kiểm định.

### Dừng sớm

Cách chống quá khớp rẻ nhất là dừng đúng lúc: giữ trọng số ở lượt có mất mát kiểm định thấp nhất. `train()` đã làm
việc đó: kỷ lục cuối cùng ở lượt 20, và sau 10 lượt không có kỷ lục mới thì nó ghi lượt dừng 30. Đến lúc này ta mới đọc
tập kiểm tra, một lần, ở trọng số đã chọn:

In [ ]:
test_best = evaluate(base["best_params"], Xte32, yte)[1]
test_final = evaluate(base["params"], Xte32, yte)[1]
print(f"dừng sớm: giữ lượt {b_ep}, dừng ở lượt {base['stop']} (chờ {PATIENCE} lượt không có kỷ lục mới)")
print(f"kiểm tra: trọng số lượt {b_ep} đúng {pct(test_best / 10000)} · học hết 150 lượt đúng {pct(test_final / 10000)}")
seeds_test = [s["testOk"] / 10000 for s in config()["seeds"]]
print(f"demo, 3 seed, trọng số lúc dừng sớm: kiểm tra đúng {pct(min(seeds_test))}–{pct(max(seeds_test))}")
assert abs(test_best - config()["testBest"]) <= 3 and abs(test_final - config()["testFinal"]) <= 3, "khác với demo"

Dừng sớm cắt được 120 lượt huấn luyện, nhưng nó tối ưu *mất mát* kiểm định chứ không phải độ chính xác: ở đây trọng số
của lượt 150 lại đúng nhiều hơn một chút trên tập kiểm tra (89,57% so với 89,45%). Khoảng chênh đó nhỏ hơn dao động
giữa các seed, nên không nói được lượt nào “tốt hơn”. Dừng sớm nhanh và rẻ, nhưng không sửa được gốc của vấn đề: 1.000 ảnh là quá ít.

## 5 · Học thuộc cả nhãn vô nghĩa

Mạng học được quy luật, hay chỉ nhớ? Zhang và cộng sự (2016) thử một cách rất thẳng: xáo ngẫu nhiên các nhãn. Ảnh vẫn
là 1.000 ảnh đó, nhưng nhãn của mỗi ảnh được chia lại ngẫu nhiên (`SHUFFLED`, ở mục 2), nên không còn quy luật nào
nối ảnh với nhãn. Chỉ có thể học thuộc.

### 🤔 Dự đoán trước
Với nhãn bị xáo, mạng có đạt 100% đúng trên tập huấn luyện không? Nếu có, nó cần nhiều lượt hơn hay ít hơn so với
nhãn thật (lượt 26)?

In [ ]:
shuf = train(labels=SHUFFLED, epochs=70)
check(shuf, runs["shuffled"])
f_shuf = first_perfect(shuf)
print(f"lần đầu đúng 100% ảnh huấn luyện: nhãn thật ở lượt {first_perfect(base)}, nhãn xáo ở lượt {f_shuf}", end="")
print(f" (chậm hơn {vn(f_shuf / first_perfect(base), 1)} lần)")
for ep in [0, 30, 70]:
    print(
        f"nhãn xáo, lượt {ep:>2}: huấn luyện đúng {pct(shuf['train'][ep][1] / 1000, 1):>6} · "
        f"kiểm định đúng {pct(shuf['val'][ep][1] / 10000)} · mất mát kiểm định {vn(shuf['val'][ep][0], 2)}"
    )
print(
    f"‖W‖² ở lượt 70: nhãn thật {vn(base['w2'][70], 1)} · nhãn xáo {vn(shuf['w2'][70], 1)} (lúc đầu {vn(base['w2'][0], 1)})"
)
assert f_shuf == 68 and shuf["val"][70][1] < 1500, "khác với demo: hãy kiểm tra lại dữ liệu"

Hình dưới so hai lần huấn luyện theo lượt: nhãn thật (xanh, liền, chấm tròn) và nhãn xáo (cam, nét gạch, chấm vuông).
Bên trái là tỉ lệ đúng trên tập huấn luyện, bên phải trên tập kiểm định; đường xám chấm là mức đoán mò, 10%.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for log, label, st in [(base, "nhãn thật", TRAIN_STYLE), (shuf, "nhãn xáo ngẫu nhiên", VAL_STYLE)]:
    n = min(len(log["train"]), 71)
    a.plot(np.arange(n), [ok / 1000 for _, ok in log["train"][:n]], markevery=5, label=label, **st)
    ep_v, ok_v = val_curve(log, 1)
    keep = [i for i, e in enumerate(ep_v) if e <= 70]
    b.plot([ep_v[i] for i in keep], [ok_v[i] / 10000 for i in keep], markevery=2, **st)
for ax, title in [(a, "Tập huấn luyện (đề cũ)"), (b, "Tập kiểm định (đề mới)")]:
    ax.axhline(0.1, color=GREY, ls=":", lw=1.2)
    ax.set(xlabel="lượt (epoch)", title=title, ylim=(0, 1.03))
    ax.grid(alpha=0.3)
a.set_ylabel("tỉ lệ đúng")
b.annotate("đoán mò: 10%", (70, 0.1), xytext=(0, 5), textcoords="offset points", ha="right", fontsize=9)
a.legend(frameon=False, fontsize=9, loc="lower right")
vn_axes(fig)
plt.tight_layout()
plt.show()

Có: lần đầu đúng cả 1.000 nhãn vô nghĩa ở lượt 68, chỉ chậm hơn nhãn thật khoảng 2,6 lần. Trên ảnh mới nó vẫn ở quanh
mức đoán mò, vì không có gì để tổng quát hóa. Mất mát huấn luyện thấp **không** chứng minh mạng đã hiểu điều gì.

Zhang và cộng sự thấy đúng điều này với các mạng tích chập lớn trên CIFAR-10 và ImageNet. Họ kết luận rằng số tham số
một mình không giải thích được vì sao mạng tổng quát hóa: cùng một mạng, cùng thuật toán, có thể học quy luật hoặc học
thuộc, tùy dữ liệu; và học thuộc nhãn ngẫu nhiên chỉ tốn thời gian hơn “một hệ số nhỏ”. Để ý thêm: học thuộc nhiễu cần trọng số lớn hơn, ‖W‖² tăng nhanh hơn hẳn. Điều đó gợi ý một cách
chống lại: phạt trọng số lớn.

## 6 · Phạt L2: kéo trọng số về 0

Ta cộng vào mất mát một khoản phạt tỉ lệ với tổng bình phương các trọng số:

$$\mathcal{L}_{\text{total}} = \mathcal{L} + \lambda \lVert W\rVert^2, \qquad
\frac{\partial\, \lambda\lVert W\rVert^2}{\partial W} = 2\lambda W$$

Như thường làm, chỉ phạt ma trận $W$, không phạt độ lệch $b$. Với SGD thuần, một bước là

$$W \leftarrow W - \eta\,(\nabla\mathcal{L} + 2\lambda W) = (1 - 2\eta\lambda)\,W - \eta\,\nabla\mathcal{L}$$

Tức là trước mỗi bước, mọi trọng số bị **co** lại theo hệ số $1 - 2\eta\lambda$: đó là **weight decay**. Ở Bài 07, AdamW
viết phép co là $1 - \eta\lambda$, nên $\lambda$ của nó ứng với $2\lambda$ ở đây. Ô dưới kiểm tra rằng hai cách viết là
một với SGD thuần, trên một batch thật, và tính xem phép co tích lũy bao nhiêu sau 150 lượt.

In [ ]:
lam = 0.003
rng_probe = np.random.default_rng(5)
W0 = [[rng_probe.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]


def copy(params):
    return [[W.copy(), b.copy()] for W, b in params]


penalized, plain = copy(W0), copy(W0)
step(penalized, Xtr[:64], ytr[:64], lam, 0, None)  # một bước trên L + λ‖W‖²
step(plain, Xtr[:64], ytr[:64], 0, 0, None)  # một bước SGD thường: W − η∇L
gap = max(float(np.abs(Wp - (Wq - 2 * ETA * lam * W)).max()) for (Wp, _), (Wq, _), (W, _) in zip(penalized, plain, W0))
print(f"bước có phạt L2 so với “bước thường rồi co thêm 2ηλW”: sai khác lớn nhất {gap:.1e}".replace(".", ","))
n_steps = EPOCHS * -(-len(Xtr) // BATCH)
shrink = (1 - 2 * ETA * lam) ** n_steps
print(f"λ = {vn(lam, 3)}: mỗi bước nhân W với 1 − 2ηλ = {vn(1 - 2 * ETA * lam, 4)}")
print(f"chỉ riêng phép co, sau {vn(n_steps, 0)} bước: W còn {vn(shrink, 3)} lần, ‖W‖² còn {vn(shrink**2, 3)} lần")
assert gap < 1e-12

Hai cách viết trùng nhau đến sai số làm tròn. Nếu chỉ có phép co, sau 150 lượt mỗi trọng số còn 0,237 lần giá trị
ban đầu, và ‖W‖² còn 0,056 lần. Gradient của dữ liệu kéo ngược lại: trọng số nào thật sự giúp giảm mất mát thì được
giữ, trọng số nào chỉ để nhớ vài ảnh thì bị co dần về 0. Ta chạy lại 150 lượt với $\lambda = 0{,}003$:

In [ ]:
l2 = train(lam=0.003, dense=40, val_every=10)
check(l2, config(lam=0.003))
for name, log in [("không phạt", base), ("λ = 0,003", l2)]:
    b_ = log["best"]
    print(
        f"{name:>10}: ‖W‖² lượt 0 · 20 · 150 = {' · '.join(vn(log['w2'][e], 1) for e in (0, 20, 150))} · "
        f"100% huấn luyện từ lượt {first_perfect(log)} · mất mát kiểm định tốt nhất {vn(log['val'][b_][0], 4)} "
        f"(lượt {b_}), lượt 150: {vn(log['val'][150][0], 4)}"
    )
assert (first_perfect(l2), l2["best"], l2["stop"]) == (40, 29, 39), "khác với demo: hãy kiểm tra lại dữ liệu"
assert l2["val"][150][0] < l2["val"][l2["best"]][0], "khác với demo: hãy kiểm tra lại dữ liệu"

Ba điều thay đổi. Trọng số đi theo hướng ngược lại: không phạt thì ‖W‖² *tăng* từ 405 lên 495, có phạt thì giảm
về 95,1 (vẫn lớn hơn mức 405 × 0,056 của riêng phép co). Mạng vẫn thuộc hết 1.000 ảnh, chỉ chậm hơn (lượt 40 thay vì
26). Và mất mát kiểm định không đi lên nữa: ở lượt 150 nó còn thấp hơn ở lượt 29, kỷ lục mà dừng sớm đã giữ. Với phạt
L2, dừng sớm dừng quá sớm.

$\lambda$ là một **siêu tham số**: phạt yếu quá thì không có tác dụng, mạnh quá thì mạng không khớp nổi cả dữ liệu
huấn luyện. Hình dưới vẽ cả 5 giá trị $\lambda$ mà demo đã thử (seed 0, từ tệp của demo; hai đường $\lambda = 0$ và
$0{,}003$ trùng với hai lần chạy vừa rồi). Trái: ‖W‖² theo lượt. Phải: mất mát kiểm định, đo mỗi lượt. Mỗi
$\lambda$ một màu và một dạng điểm.

In [ ]:
LAM_STYLE = {
    0: (GREY, "o", "-"),
    0.0003: (BLUE, "v", (0, (1, 1.5))),
    0.001: (BLUE, "s", (0, (6, 3))),
    0.003: (GREEN, "^", "-"),
    0.01: (ORANGE, "D", (0, (3, 1, 1, 1))),
}
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
for lam_, (color, mk, ls) in LAM_STYLE.items():
    c = config(lam=lam_)
    kw = {"color": color, "marker": mk, "ls": ls, "ms": 4, "lw": 1.5, "markevery": 15}
    a.plot(ep_all, c["w2"], label=f"λ = {vn_tick(lam_)}", **kw)
    b.plot(ep_all, c["valLoss"], **kw)
a.set(xlabel="lượt (epoch)", ylabel="‖W‖²", title="Độ lớn trọng số")
b.set(xlabel="lượt (epoch)", ylabel="mất mát kiểm định", ylim=(0.3, 0.55), title="Mất mát kiểm định")
for ax in (a, b):
    ax.grid(alpha=0.3)
a.legend(frameon=False, fontsize=9, ncol=2)
vn_axes(fig)
plt.tight_layout()
plt.show()
c01 = config(lam=0.01)
print(
    f"λ = 0,01: huấn luyện đúng {pct(c01['trainOk'][-1] / 1000, 1)} ở lượt 150; mất mát kiểm định tốt nhất {vn(c01['valLoss'][c01['best']], 4)}"
)

Phạt càng mạnh, ‖W‖² càng nhỏ, và đường mất mát kiểm định càng phẳng ở cuối. Nhưng $\lambda = 0{,}01$ (cam, hình thoi)
đi quá đà: trọng số bị ép nhỏ tới mức mạng không còn đúng hết tập huấn luyện, và mất mát kiểm định tốt nhất của nó cao
hơn cả khi không phạt. Chọn $\lambda$ thế nào? Bằng tập kiểm định, ở mục 9.

## 7 · Dropout: mỗi ảnh, mỗi bước một mạng con khác nhau

Dropout (Hinton và cộng sự, 2012; Srivastava và cộng sự, 2014) tắt ngẫu nhiên một phần nơ-ron ẩn mỗi khi tính xuôi lúc
huấn luyện. Mỗi nơ-ron bị tắt với xác suất $p$, độc lập với các nơ-ron khác và với các ảnh khác:

$$\tilde h = \frac{m \odot h}{1-p}, \qquad m_i \sim \operatorname{Bernoulli}(1-p)$$

Chia cho $1 - p$ để kỳ vọng không đổi: $\mathbb{E}[\tilde h_i] = (1-p)\cdot\frac{h_i}{1-p} = h_i$. Nhờ vậy khi suy
luận ta dùng đủ cả mạng mà không phải sửa gì (cách viết này gọi là *inverted dropout*, cũng là cách của PyTorch). Lưu ý:
Srivastava và cộng sự dùng $p$ cho xác suất **giữ lại** một nơ-ron, ngược với ở đây và với PyTorch.

Lan truyền ngược qua dropout cũng đơn giản: nơ-ron bị tắt không truyền gradient, nơ-ron còn lại truyền gradient nhân
$1/(1-p)$. Đó là lý do `step()` gộp mặt nạ dropout với đạo hàm ReLU vào cùng một mảng `m1`, `m2`.

In [ ]:
def dropout_mask(shape, p, rng):
    """Mặt nạ inverted dropout: 0 với xác suất p, còn lại 1/(1 − p). Mỗi hàng (mỗi ảnh) một mặt nạ riêng."""
    return (rng.random(shape) >= p) / (1 - p)


demo_rng = np.random.default_rng(3)
m = dropout_mask((64, 128), 0.2, demo_rng)  # một batch 64 ảnh, lớp ẩn 128 nơ-ron
print(f"tỉ lệ nơ-ron bị tắt trong batch: {pct((m == 0).mean())} · giá trị của nơ-ron còn lại: {vn(m.max(), 2)}")
print(
    f"trung bình của mặt nạ: {vn(m.mean(), 3)} (kỳ vọng đúng bằng 1) · số mặt nạ khác nhau trong 64 ảnh: {len({r.tobytes() for r in m})}"
)
big = dropout_mask((1000, 128), 0.2, demo_rng)
print(f"trên 1.000 × 128 lần rút: trung bình {vn(big.mean(), 4)}")
assert abs(big.mean() - 1) < 0.01

Khoảng 20% nơ-ron bị tắt, nơ-ron còn lại được nhân 1,25, và trung bình của mặt nạ gần đúng bằng 1. Mỗi ảnh trong batch
có mặt nạ riêng, và `train()` rút mặt nạ mới ở mỗi bước. Nên trong 150 lượt, mạng không bao giờ thấy cùng một mạng con
hai lần: mỗi nơ-ron phải hữu ích dù những nơ-ron khác có mặt hay không, thay vì dựa vào một nhóm cụ thể để cùng nhớ
một ảnh (Hinton và cộng sự gọi đó là tránh “đồng thích nghi”, co-adaptation).

### 🤔 Dự đoán trước
Phạt L2 chống quá khớp bằng cách giữ trọng số nhỏ. Với dropout $p = 0{,}2$ trên cả hai lớp ẩn (không phạt L2), sau 150
lượt ‖W‖² nhỏ hơn hay lớn hơn khi không dùng dropout (495)?

In [ ]:
drop = train(p=0.2, dense=40, val_every=10)
check(drop, config(p=0.2))
for name, log in [("không dropout", base), ("dropout 0,2", drop)]:
    b_ = log["best"]
    print(
        f"{name:>13}: ‖W‖² lượt 150 = {vn(log['w2'][150], 1)} · 100% huấn luyện từ lượt {first_perfect(log)} · "
        f"mất mát kiểm định tốt nhất {vn(log['val'][b_][0], 4)} (lượt {b_}) → lượt 150: {vn(log['val'][150][0], 4)}"
    )
assert (first_perfect(drop), drop["best"], drop["stop"]) == (38, 28, 38), "khác với demo: hãy kiểm tra lại dữ liệu"
print(f"demo, 3 seed, kiểm định đúng lúc dừng sớm: không dropout {fmt_range(config(), 'valOk')}", end=" · ")
print(f"dropout 0,2 {fmt_range(config(p=0.2), 'valOk')}")
assert drop["w2"][150] > base["w2"][150], "khác với demo: hãy kiểm tra lại dữ liệu"

Lớn hơn: dropout không làm trọng số nhỏ đi, thậm chí ngược lại. Nó chống quá khớp theo cách khác: khó thuộc lòng hơn
(đúng hết tập huấn luyện ở lượt 38 thay vì 26), và mất mát kiểm định tốt nhất thấp hơn rõ (0,3287 so với 0,3664). Ở
điểm dừng sớm, cả 3 seed có dropout đều đúng nhiều hơn cả 3 seed không dropout: khác biệt nằm ngoài dao động giữa các
seed. Nhưng dropout không ngăn được quá khớp: sau lượt 28, mất mát kiểm định vẫn đi lên. Dropout và phạt L2 bổ sung cho nhau; mục 9 sẽ thử kết hợp.

Khi suy luận, `evaluate()` dùng đủ cả mạng, không mặt nạ. Có thể làm vậy là nhờ phép chia $1/(1-p)$ lúc huấn luyện.
Ô dưới kiểm tra điều đó trên mạng vừa học: lấy trung bình đầu ra lớp ẩn thứ nhất qua 200 mặt nạ, so với đầu ra không
mặt nạ.

In [ ]:
W1, b1 = drop["params"][0]
h1 = np.maximum(0, Xval[:200] @ W1 + b1)  # đầu ra lớp ẩn 1 của 200 ảnh kiểm định, chế độ suy luận
avg = np.mean([h1 * dropout_mask(h1.shape, 0.2, demo_rng) for _ in range(200)], axis=0)
rel = np.abs(avg - h1).sum() / h1.sum()
print(f"trung bình qua 200 mặt nạ so với không mặt nạ: sai khác tương đối {pct(rel)}")
no_scale = np.mean([h1 * (demo_rng.random(h1.shape) >= 0.2) for _ in range(200)], axis=0)
print(f"nếu quên chia cho 1 − p: trung bình chỉ bằng {pct(no_scale.sum() / h1.sum(), 1)} đầu ra thật")

Có chia cho $1 - p$ thì trung bình qua nhiều mặt nạ rất gần đầu ra của cả mạng (sai khác còn lại là do chỉ rút 200
mặt nạ). Quên chia thì khi huấn luyện, lớp sau chỉ quen với 80,0% tín hiệu, và lúc suy luận mỗi nơ-ron lại lớn gấp
1,25 lần mức nó đã quen. (Với lớp ẩn thứ hai, trung bình qua mặt nạ chỉ *xấp xỉ*
mạng đầy đủ, vì ReLU ở giữa không tuyến tính. Srivastava và cộng sự gọi cách dùng cả mạng khi suy luận là một phép xấp
xỉ của việc lấy trung bình mọi mạng con, và nó chạy tốt trong thực tế.)

## 8 · Tăng cường dữ liệu: cùng 1.000 ảnh, mỗi lượt một vị trí khác

Cách tốt nhất để chống quá khớp là có thêm dữ liệu thật. Gần tốt như thế là tạo dữ liệu “nửa thật” từ dữ liệu có sẵn,
bằng những phép biến đổi không đổi nhãn: một chữ 3 dịch sang trái 2 điểm ảnh vẫn là chữ 3. Hàm dưới (của
`overfit.py`) dịch mỗi ảnh một số nguyên điểm ảnh ngẫu nhiên trong $[-2, 2]$ theo mỗi chiều, phần trống điền 0.

In [ ]:
def shift_images(x, rng):
    """Dịch mỗi ảnh một số nguyên điểm ảnh ngẫu nhiên trong [−SHIFT, SHIFT] theo mỗi chiều (điền 0)."""
    n = len(x)
    pad = np.zeros((n, 28 + 2 * SHIFT, 28 + 2 * SHIFT))
    pad[:, SHIFT : SHIFT + 28, SHIFT : SHIFT + 28] = x.reshape(n, 28, 28)
    d = rng.integers(0, 2 * SHIFT + 1, (n, 2))  # góc trên trái của khung 28×28 trong ảnh đã đệm
    return np.stack([pad[k, d[k, 0] : d[k, 0] + 28, d[k, 1] : d[k, 1] + 28] for k in range(n)]).reshape(n, 784)


views = shift_images(np.repeat(Xtr[:1], 8, axis=0), np.random.default_rng(4))
fig, axes = plt.subplots(1, 9, figsize=(10, 1.5))
for ax, img, title in zip(axes, [Xtr[0], *views], ["gốc", *[f"lần {k}" for k in range(1, 9)]]):
    ax.imshow(img.reshape(28, 28), cmap="gray_r")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.show()
print(
    f"{(2 * SHIFT + 1) ** 2} vị trí có thể cho mỗi ảnh; 1.000 ảnh × {(2 * SHIFT + 1) ** 2} = {vn(1000 * (2 * SHIFT + 1) ** 2, 0)} ảnh khác nhau"
)

Hình trên là ảnh huấn luyện đầu tiên và 8 lần nó được dịch ngẫu nhiên. Mạng MLP không biết hai ảnh lệch nhau 1 điểm
ảnh là “cùng một hình”: với nó, mỗi vị trí là 784 số khác hẳn. Vì vậy nó khó thuộc lòng hơn nhiều. Ta chạy 150 lượt,
đo tập kiểm định mỗi 5 lượt:

In [ ]:
aug = train(aug=True)
check(aug, config(aug=True))
ca = config(aug=True)
for name, log, c in [("không dịch", base, config()), ("dịch ±2 px", aug, ca)]:
    print(
        f"{name:>10}: lượt 150 huấn luyện đúng {pct(log['train'][150][1] / 1000, 1)}, kiểm định đúng "
        f"{pct(log['val'][150][1] / 10000)} · dừng sớm (theo demo) giữ lượt {c['best']}: kiểm định "
        f"{vn(c['valLoss'][c['best']], 4)}, kiểm tra đúng {pct(c['testBest'] / 10000)}"
    )
print(
    f"dịch ±2 px: lần đầu đúng cả 1.000 ảnh huấn luyện ở lượt {first_perfect(aug)} (không dịch: lượt {first_perfect(base)})"
)
assert first_perfect(aug) == 130 and aug["val"][150][1] > 9300, "khác với demo: hãy kiểm tra lại dữ liệu"

Đây là bước nhảy lớn nhất trong cả notebook: độ chính xác kiểm định ở lượt 150 từ 89,89% lên 94,10%, mất mát kiểm
định tốt nhất giảm gần một nửa (0,3664 → 0,1980). Mạng vẫn *có thể* thuộc lòng (nó chạm 100% ảnh huấn luyện một lần, ở lượt 130), nhưng phải mất
gấp 5 lần số lượt, vì mỗi lượt nó thấy các ảnh ở vị trí mới. Phép dịch dạy mạng một điều đúng về chữ viết tay mà 1.000
ảnh không tự nói ra: vị trí của nét không quan trọng bằng hình dạng của nó.

## 9 · Chọn trên tập kiểm định, báo cáo trên tập kiểm tra

Ta đã có bốn cách: dừng sớm, phạt L2, dropout, dịch ảnh. Kết hợp thế nào là tốt nhất? Demo đã chạy cả lưới
$\lambda \in \{0;\ 0{,}0003;\ 0{,}001;\ 0{,}003;\ 0{,}01\}$ × $p \in \{0;\ 0{,}2;\ 0{,}5\}$ × có/không dịch ảnh, mỗi tổ
hợp 3 seed, 150 lượt, với dừng sớm. Bốn lần chạy ở trên là bốn ô seed 0 của lưới đó, và đều khớp. Phần còn lại mất vài
phút, nên ta đọc kết quả mà demo đã lưu.

Quy tắc chọn: tổ hợp có **độ chính xác kiểm định cao nhất** ở trọng số dừng sớm, trung bình 3 seed. Tập kiểm tra không
được dùng để chọn; ta chỉ đọc nó ở tổ hợp đã chọn.

In [ ]:
def label(c):
    return f"λ = {vn_tick(c['lam'])}, p = {vn_tick(c['p'])}, {'dịch ảnh' if c['aug'] else 'không dịch'}"


cfgs = runs["configs"]
chosen = max(cfgs, key=lambda c: seed_stats(c, "valOk")[0])
best_plain = max((c for c in cfgs if not c["aug"]), key=lambda c: seed_stats(c, "valOk")[0])
print(f"{len(cfgs)} tổ hợp × {len(runs['seeds'])} seed. Kiểm định đúng (trung bình, min–max 3 seed) → kiểm tra đúng:")
for name, c in [("không chính quy hóa", config()), ("tốt nhất khi không dịch ảnh", best_plain), ("tốt nhất", chosen)]:
    print(f"  {name:>27}: {label(c):<36} kiểm định {fmt_range(c, 'valOk')} → kiểm tra {fmt_range(c, 'testOk')}")
ranked = sorted(cfgs, key=lambda c: -seed_stats(c, "valOk")[0])
second = ranked[1]
overlap = seed_stats(second, "valOk")[2] >= seed_stats(chosen, "valOk")[1]
print(
    f"  {'đứng thứ hai':>27}: {label(second):<36} kiểm định {fmt_range(second, 'valOk')}; chồng lên tốt nhất: {overlap}"
)
n_better = sum(seed_stats(c, "valOk")[0] > seed_stats(best_plain, "valOk")[0] for c in cfgs if c["aug"])
print(f"số tổ hợp có dịch ảnh tốt hơn tổ hợp tốt nhất không dịch: {n_better}/{sum(c['aug'] for c in cfgs)}")
for aug_on in [False, True]:
    wins = sum(
        seed_stats(config(l_, 0.5, aug_on), "valOk")[0] > seed_stats(config(l_, 0, aug_on), "valOk")[0]
        for l_ in runs["lambdas"]
    )
    c5, c0 = config(0, 0.5, aug_on), config(0, 0, aug_on)
    print(
        f"{'có' if aug_on else 'không'} dịch ảnh: p = 0,5 tốt hơn p = 0 ở {wins}/{len(runs['lambdas'])} giá trị λ · "
        f"λ = 0, lượt 150: huấn luyện đúng {pct(c5['trainOk'][-1] / 1000, 1)} (p = 0,5), {pct(c0['trainOk'][-1] / 1000, 1)} (p = 0)"
    )
full = runs["full"]
print(
    f"so sánh: cùng mạng học đủ {vn(full['nTrain'], 0)} ảnh trong {full['epochs']} lượt: kiểm tra đúng {pct(full['testAcc'])}"
)

Hình dưới vẽ cả lưới: độ chính xác kiểm định (trung bình 3 seed, vạch là min–max) theo $\lambda$, trái là không dịch
ảnh, phải là có dịch ảnh. Mỗi $p$ một màu, một dạng điểm và một kiểu nét. Ngôi sao đen là tổ hợp được chọn.

In [ ]:
P_STYLE = {0: (GREY, "o", "-"), 0.2: (BLUE, "s", (0, (6, 3))), 0.5: (ORANGE, "^", (0, (1, 1.5)))}
lams = runs["lambdas"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, aug_on in zip(axes, [False, True]):
    for p_, (color, mk, ls) in P_STYLE.items():
        st = [seed_stats(config(lam_, p_, aug_on), "valOk") for lam_ in lams]
        m = np.array([s[0] for s in st])
        err = [m - [s[1] for s in st], [s[2] for s in st] - m]
        x = np.arange(len(lams)) + (p_ - 0.25) * 0.25  # lệch nhẹ để các vạch không đè nhau
        ax.errorbar(
            x, m, yerr=err, color=color, marker=mk, ls=ls, ms=6, lw=1.5, capsize=3, label=f"dropout p = {vn_tick(p_)}"
        )
    if chosen["aug"] == aug_on:
        k = lams.index(chosen["lam"])
        ax.plot(k + (chosen["p"] - 0.25) * 0.25, seed_stats(chosen, "valOk")[0], "*", color="black", ms=15, zorder=5)
    ax.set_xticks(range(len(lams)), [vn_tick(v) for v in lams])
    ax.set(xlabel="phạt L2: λ", title="Có dịch ảnh ±2 px" if aug_on else "Không dịch ảnh")
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("kiểm định đúng (dừng sớm)")
axes[0].legend(frameon=False, fontsize=9, loc="lower left")
vn_axes(fig)
for ax in axes:
    ax.set_xticks(range(len(lams)), [vn_tick(v) for v in lams])  # giữ nhãn λ dạng 0,0003 thay vì vị trí 0…4
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.tight_layout()
plt.show()

Tổ hợp được chọn kết hợp cả ba: phạt L2 nhẹ, dropout 0,2 và dịch ảnh. Nó cũng là tổ hợp tốt nhất trên tập kiểm tra,
nhưng ta không dùng điều đó để chọn nó. Ba điều đọc được từ hình:
- **Dịch ảnh thắng rõ nhất.** 14/15 tổ hợp có dịch ảnh tốt hơn tổ hợp tốt nhất không dịch ảnh, vượt xa dao động giữa
  các seed. Không nút vặn nào trên trọng số làm được điều đó.
- **Mạnh quá thì hại.** $\lambda = 0{,}01$ kém nhất ở mọi $p$, có hay không dịch ảnh. Dropout $p = 0{,}5$ giúp ở 4/5
  giá trị $\lambda$ khi không dịch ảnh, nhưng khi đã dịch ảnh thì nó kém hơn không dropout ở mọi $\lambda$: hai nguồn
  nhiễu cộng lại là quá nhiều cho mạng nhỏ này, nó không còn khớp hết dữ liệu huấn luyện (96,3% so với 99,9%).
- **Ở đỉnh, khác biệt nằm trong dao động giữa các seed.** Khoảng 3 seed của tổ hợp được chọn chồng lên khoảng của tổ
  hợp đứng thứ hai, nên “tốt nhất” ở đây chỉ hơn hàng xóm của nó một chút.

Vẫn còn một khoảng cách lớn: dù chính quy hóa tốt nhất, 1.000 ảnh cho 94,78% trên tập kiểm tra, còn mạng học đủ
60.000 ảnh (Bài 04) đạt 97,27%. Thêm dữ liệu thật vẫn là cách chống quá khớp tốt nhất.

## 10 · Lớn hơn nữa thì sao? Double descent

Bức tranh “mô hình càng lớn càng dễ quá khớp” gợi ý một đường cong chữ U: lỗi kiểm định giảm khi mô hình lớn dần, rồi
tăng khi nó đủ lớn để thuộc lòng. Để quét kích thước mà không phải huấn luyện hàng chục mạng, demo dùng một mô hình đơn
giản hơn trên cùng 1.000 ảnh (Belkin và cộng sự, 2019):

$$\phi(x) = \mathrm{ReLU}(xV), \qquad \hat y = \phi(x)\,W$$

$V$ là ma trận 784 × N **ngẫu nhiên, giữ cố định** (mỗi cột chuẩn hóa về độ dài 1); chỉ học $W$ (N × 10), bằng bình
phương tối thiểu trên nhãn one-hot $Y$. Gọi $F = \phi(X_{\text{train}})$ là ma trận đặc trưng 1.000 × N. Không cần
gradient, vì có lời giải đúng:
- **N < 1.000:** ít tham số hơn số ảnh, nên thường không khớp hết; $W = (F^\top F)^{-1} F^\top Y$ là cách khớp tốt nhất.
- **N ≥ 1.000:** có vô số $W$ khớp đúng mọi ảnh; $W = F^\top (F F^\top)^{-1} Y$ là cách có $\lVert W\rVert$ **nhỏ
  nhất** (giống `np.linalg.lstsq`).
- **Có phạt L2:** $W = F^\top (F F^\top + \lambda I)^{-1} Y$ cực tiểu $\lVert FW - Y\rVert^2 + \lambda\lVert W\rVert^2$,
  với mọi N. Demo dùng $\lambda = 1$.

Như `overfit.py`, N đặc trưng là N cột đầu của cùng một ma trận ngẫu nhiên (seed 7), nên mô hình lớn chứa mô hình nhỏ.
Notebook quét N đến 3.000; phần còn lại của đường cong (tới N = 20.000) lấy từ tệp của demo, vì ma trận đặc trưng của
tập kiểm định khi đó quá lớn cho 2 nhân CPU.

### 🤔 Dự đoán trước
Ô dưới in lỗi kiểm định ở N = 400 (ít tham số hơn số ảnh) và N = 3.000 (nhiều gấp ba). Ở N = 1.000, đúng bằng số ảnh
huấn luyện, lỗi kiểm định nằm giữa hai mức đó, thấp hơn cả hai, hay cao hơn cả hai?

In [ ]:
WIDTHS = [w for w in runs["doubleDescent"]["widths"] if w <= 3000]
V_all = np.random.default_rng(7).normal(size=(784, runs["doubleDescent"]["widths"][-1]))  # cùng số ngẫu nhiên với demo
V = V_all[:, : WIDTHS[-1]] / np.linalg.norm(V_all[:, : WIDTHS[-1]], axis=0)
print(f"ma trận ngẫu nhiên 784 × {vn(V_all.shape[1], 0)} như demo; notebook dùng {vn(V.shape[1], 0)} cột đầu")
F_tr = np.maximum(0, Xtr @ V)  # đặc trưng của 1.000 ảnh huấn luyện; N đặc trưng = N cột đầu
F_val = np.maximum(0, Xval32 @ V.astype(np.float32))  # của 10.000 ảnh kiểm định (float32 cho nhanh)
Y = np.eye(10)[ytr]  # nhãn one-hot


def fit_last_layer(F, ridge=0.0):
    """W cho ŷ = F W: bình phương tối thiểu, nghiệm khớp đúng có ‖W‖ nhỏ nhất, hoặc có phạt L2 (ridge > 0)."""
    n_img, n = F.shape
    if n < n_img and not ridge:
        return np.linalg.solve(F.T @ F, F.T @ Y)
    return F.T @ np.linalg.solve(F @ F.T + ridge * np.eye(n_img), Y)


def n_wrong(F, W, y):
    return int(((F @ W.astype(F.dtype)).argmax(1) != y).sum())


dd = {}
for n in WIDTHS:
    W, W_r = fit_last_layer(F_tr[:, :n]), fit_last_layer(F_tr[:, :n], ridge=runs["doubleDescent"]["ridge"])
    errs = [n_wrong(F[:, :n], w, y) for w in (W, W_r) for F, y in ((F_tr, ytr), (F_val, yval))]
    dd[n] = {"err": errs, "norm": float(np.linalg.norm(W))}
for n in [100, 400, 900, 1000, 1100, 3000]:
    tr, va, _, va_r = dd[n]["err"]
    print(
        f"N = {vn(n, 0):>5}: lỗi huấn luyện {pct(tr / 1000, 1):>5} · lỗi kiểm định {pct(va / 10000, 1):>5} · "
        f"‖W‖ = {vn(dd[n]['norm'], 1):>7} · có phạt L2: lỗi kiểm định {pct(va_r / 10000, 1)}"
    )
ref = runs["doubleDescent"]
for k, n in enumerate(WIDTHS):
    want = [ref[m][e][k] for m in ("minNorm", "withRidge") for e in ("trainErr", "valErr")]
    assert max(abs(a - b) for a, b in zip(dd[n]["err"], want)) <= 3, f"khác với demo ở N = {n}"
peak = dd[1000]["err"][1]
assert peak > dd[400]["err"][1] and peak > dd[3000]["err"][1] and dd[1000]["err"][3] < peak
print("trùng với demo ở mọi N ≤ 3.000")

Cao hơn cả hai. Hình dưới vẽ cả đường cong, trục N thang log: lỗi kiểm định không phạt (cam, nét gạch, vuông), có phạt
L2 $\lambda = 1$ (xanh lá, liền, tam giác) và lỗi huấn luyện không phạt (xanh, chấm, tròn). Điểm tô đặc là do notebook
tính (N ≤ 3.000), điểm rỗng lấy từ tệp của demo. Vạch xám đứng là N = 1.000, đúng bằng số ảnh huấn luyện.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
N_all = np.array(ref["widths"])
series = [
    ("lỗi kiểm định, không phạt", ref["minNorm"]["valErr"], 10000, 1, ORANGE, "s", (0, (6, 3))),
    ("lỗi kiểm định, phạt L2 (λ = 1)", ref["withRidge"]["valErr"], 10000, 3, GREEN, "^", "-"),
    ("lỗi huấn luyện, không phạt", ref["minNorm"]["trainErr"], 1000, 0, BLUE, "o", (0, (1, 1.5))),
]
for name, stored, total, k, color, mk, ls in series:
    ax.plot(N_all, np.array(stored) / total, color=color, ls=ls, lw=1.6, label=name)
    live = np.array([dd[n]["err"][k] for n in WIDTHS]) / total
    ax.plot(WIDTHS, live, mk, color=color, ms=6)
    far = N_all > WIDTHS[-1]
    ax.plot(N_all[far], np.array(stored)[far] / total, mk, color=color, mfc="white", ms=6)
ax.axvline(len(Xtr), color=GREY, ls=":", lw=1.2)
ax.annotate("N = số ảnh huấn luyện", (len(Xtr), 0.95), xytext=(6, 0), textcoords="offset points", fontsize=9)
ax.set(xscale="log", xlabel="số đặc trưng ngẫu nhiên N (thang log)", ylabel="tỉ lệ lỗi", ylim=(0, 1))
ax.legend(frameon=False, fontsize=9, loc="upper right", bbox_to_anchor=(1, 0.9))
ax.grid(alpha=0.3, which="both")
vn_axes(fig)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.show()
classic = min(
    (n for n in ref["widths"] if n < len(Xtr)), key=lambda n: ref["minNorm"]["valErr"][ref["widths"].index(n)]
)
k_c, k_last = ref["widths"].index(classic), len(ref["widths"]) - 1
print(f"tốt nhất trước đỉnh: N = {vn(classic, 0)}, lỗi kiểm định {pct(ref['minNorm']['valErr'][k_c] / 10000, 1)}")
print(
    f"đỉnh: N = 1.000, {pct(dd[1000]['err'][1] / 10000, 1)} · ‖W‖ = {vn(dd[1000]['norm'], 1)} (N = 3.000: {vn(dd[3000]['norm'], 1)})"
)
print(
    f"N = {vn(ref['widths'][-1], 0)} (tệp của demo): không phạt {pct(ref['minNorm']['valErr'][k_last] / 10000, 1)}, "
    f"phạt L2 {pct(ref['withRidge']['valErr'][k_last] / 10000, 1)}"
)

Đường cong có hai lần giảm, nên gọi là **double descent** (Belkin và cộng sự, 2019):
- **Vùng cổ điển (N < 1.000):** lỗi kiểm định giảm rồi tăng, đúng hình chữ U quen thuộc; thấp nhất ở N = 400.
- **Đỉnh (N = 1.000):** lớp cuối vừa đủ tham số để khớp đúng mọi ảnh, và chỉ có *một* cách khớp. Ma trận $F$ gần suy
  biến, nên cách khớp duy nhất đó cần trọng số rất lớn ($\lVert W\rVert$ = 1.052,9 thay vì vài chục), và trên ảnh
  mới nó sai 87,9%, tệ hơn cả đoán mò.
- **Vùng quá tham số hóa (N > 1.000):** có vô số cách khớp đúng, và lời giải có $\lVert W\rVert$ nhỏ nhất ngày càng
  “mượt”. Lỗi kiểm định giảm lần thứ hai, xuống dưới cả điểm tốt nhất của vùng cổ điển.

Phạt L2 vừa phải ($\lambda = 1$) làm đỉnh gần như biến mất: nó không cho $W$ lớn lên, nên không có kích thước nào là
“tai họa”. Nakkiran và cộng sự (2020) chứng minh điều này cho một số mô hình hồi quy tuyến tính: phạt L2 được chỉnh
tối ưu làm lỗi kiểm định giảm đều theo kích thước mô hình.

Với mạng MLP đầy đủ như ở các mục trên, đỉnh này thường yếu hơn nhiều và rõ nhất khi nhãn có nhiễu (Nakkiran và cộng
sự, 2019). Điều cần nhớ không phải là “mô hình to luôn tốt hơn”, mà là: *số tham số so với số dữ liệu* không
một mình quyết định quá khớp; cách huấn luyện chọn nghiệm nào trong vô số nghiệm khớp mới quan trọng.

## 11 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Chính quy hóa có ngăn được việc học thuộc nhãn vô nghĩa không?
Zhang và cộng sự thấy rằng weight decay và dropout không ngăn được mạng thuộc nhãn ngẫu nhiên. Ô dưới thử dịch ảnh
trên mạng của ta, 150 lượt, nhãn xáo. Thêm `("phạt L2", {"lam": 3e-3})` hoặc `("dropout", {"p": 0.2})` vào danh sách
để thử hai cách kia (mỗi lần chạy vài giây).

In [ ]:
for name, kw in [("dịch ảnh ±2 px", {"aug": True})]:
    log = train(labels=SHUFFLED, val_every=EPOCHS, **kw)
    first = first_perfect(log)
    print(
        f"nhãn xáo + {name:<18}: lần đầu đúng 100% ảnh huấn luyện: {first if first is not None else 'chưa'} · "
        f"lượt 150: huấn luyện đúng {pct(log['train'][150][1] / 1000, 1)}, kiểm định đúng {pct(log['val'][150][1] / 10000)}"
    )

Dịch ảnh làm việc học thuộc khó hơn hẳn: sau 150 lượt mạng mới thuộc được 38,5% nhãn vô nghĩa (không dịch: đã thuộc
hết từ lượt 68), vì mỗi lượt nó phải nhớ một bộ ảnh mới. Dù vậy, kiểm định vẫn ở mức đoán mò: chính quy hóa không
biến nhiễu thành quy luật.

### Độ kiên nhẫn của dừng sớm
Demo chờ 10 lượt. Chờ ít hơn thì dừng sớm hơn và rẻ hơn, nhưng dễ dừng ở một chỗ trũng tạm thời. Ô dưới tính lại dừng
sớm trên đường cong kiểm định (đo mỗi lượt) của cả 30 tổ hợp seed 0 trong tệp của demo, với các độ kiên nhẫn khác:

In [ ]:
def early_stop(val_loss, patience):
    best = 0
    for e in range(1, len(val_loss)):
        if val_loss[e] < val_loss[best]:
            best = e
        elif e - best >= patience:
            return best, e
    return best, len(val_loss) - 1


for patience in [3, 5, 10, 20, 40]:
    picks = [early_stop(c["valLoss"], patience) for c in cfgs]
    print(
        f"chờ {patience:>2} lượt: dừng trung bình ở lượt {vn(np.mean([s for _, s in picks]), 0):>3} · "
        f"kiểm định đúng ở trọng số giữ lại: {pct(np.mean([c['valOk'][b] / 10000 for c, (b, _) in zip(cfgs, picks)]))}"
    )

Chờ 3 lượt rẻ nhất nhưng giữ lại trọng số kém hơn hẳn; từ 10 lượt trở lên thì lợi thêm rất ít. Đây cũng là một siêu
tham số, và cũng chọn trên tập kiểm định.

### Một ma trận đặc trưng ngẫu nhiên khác
Đỉnh ở N = 1.000 có phải do may rủi của seed 7? Đổi seed của $V$ và tính lại lỗi kiểm định ở N = 400, 1.000 và 1.100:

In [ ]:
print(f"seed 7 (mục 10): {' · '.join(pct(dd[n]['err'][1] / 10000, 1) for n in (400, 1000, 1100))}")
for v_seed in [8, 9]:
    Vs = np.random.default_rng(v_seed).normal(size=(784, 20000))[:, :1100]  # 1.100 cột đầu, cách rút như demo
    Vs /= np.linalg.norm(Vs, axis=0)
    Fs_tr, Fs_val = np.maximum(0, Xtr @ Vs), np.maximum(0, Xval32 @ Vs.astype(np.float32))
    errs = [n_wrong(Fs_val[:, :n], fit_last_layer(Fs_tr[:, :n]), yval) / 10000 for n in (400, 1000, 1100)]
    print(f"seed {v_seed}: {' · '.join(pct(e, 1) for e in errs)}")

Với cả ba ma trận ngẫu nhiên, lỗi kiểm định ở N = 1.000 cao gấp nhiều lần ở hai bên: đỉnh do N = số ảnh gây ra,
không phải do may rủi của một seed.

## 12 · Giới hạn và cầu nối sang bài sau

Mọi thứ ở đây đều chạy trên một mạng nông: hai lớp ẩn, và mạng *học được* dễ dàng. Vấn đề duy nhất là nó học quá kỹ.
Khi mạng sâu hơn nhiều, một vấn đề khác xuất hiện trước cả quá khớp: mạng không học được. Tín hiệu và gradient đi qua
hàng chục lớp có thể nhỏ dần về 0 hoặc lớn vô hạn, và khi đó không phạt L2 hay dropout nào giúp được. Ta cũng đã dùng
khởi tạo He từ đầu khóa mà chưa giải thích vì sao. Cả hai là chủ đề của Bài 09.

Một lưu ý về các mô hình ngôn ngữ lớn: chúng thường chỉ đọc mỗi văn bản một hoặc vài lần, nên ít có cơ hội học thuộc.
Vì vậy nhiều mô hình được tiền huấn luyện **không** dùng dropout: PaLM ghi rằng nó “được huấn luyện không dropout, dù
dropout 0,1 được dùng khi tinh chỉnh trong hầu hết trường hợp” (Chowdhery và cộng sự, 2022, mục 5). GPT đời đầu vẫn
dùng dropout 0,1 (Radford và cộng sự, 2018). Weight decay thì vẫn thường có mặt, ví dụ AdamW với $\lambda = 0{,}1$ của
LLaMA (Bài 07).

## Tóm tắt
- Trên 1.000 ảnh, mạng 109.386 tham số đúng hết tập huấn luyện từ lượt 26; mất mát kiểm định thấp nhất ở lượt 20
  rồi tăng 30% đến lượt 150, trong khi độ chính xác kiểm định gần như đứng yên: mạng sai *tự tin hơn*, không sai nhiều
  hơn.
- Xáo ngẫu nhiên nhãn, mạng vẫn thuộc hết 1.000 cặp vô nghĩa (lượt 68), và đoán mò trên ảnh mới: mất mát huấn luyện thấp
  không chứng minh mạng đã hiểu.
- Phạt L2 $\lambda\lVert W\rVert^2$ với SGD thuần là weight decay: mỗi bước nhân $W$ với $1 - 2\eta\lambda$. Với
  $\lambda = 0{,}003$, ‖W‖² giảm về 95,1 thay vì tăng lên 495, và mất mát kiểm định không đi lên nữa.
- Dropout tắt ngẫu nhiên nơ-ron, mỗi ảnh một mặt nạ, mỗi bước một mặt nạ mới, và chia cho $1 - p$ để khi suy luận dùng đủ
  cả mạng. Nó không làm trọng số nhỏ đi, nhưng làm mạng khó thuộc lòng hơn.
- Dịch ảnh ±2 px giúp nhiều hơn mọi tổ hợp λ, p: kiểm định ở lượt 150 từ 89,89% lên 94,10%. Chọn trên tập kiểm định, tổ
  hợp tốt nhất (λ = 0,001, p = 0,2, dịch ảnh) đạt 94,78% trên tập kiểm tra, vẫn kém xa 97,27% của 60.000 ảnh.
- Double descent: với đặc trưng ngẫu nhiên, lỗi kiểm định vọt lên 87,9% đúng ở N = số ảnh, rồi giảm lần thứ hai khi N
  lớn hơn; phạt L2 vừa phải xóa gần hết đỉnh.

## Câu hỏi ôn tập
1. Mất mát kiểm định tăng 30% từ lượt 20 đến lượt 150, mà độ chính xác kiểm định không giảm. Điều gì đã thay đổi?
2. Vì sao phải chọn $\lambda$, $p$ và lượt dừng trên tập kiểm định mà không phải trên tập kiểm tra?
3. Với SGD thuần, viết một bước cập nhật có phạt L2 thành “co rồi bước”. Hệ số co là bao nhiêu? Với Adam thì vì sao
   không còn như vậy?
4. Khi suy luận, vì sao dùng đủ cả mạng mà không cần sửa trọng số? Điều gì xảy ra nếu quên chia cho $1 - p$ lúc huấn
   luyện?
5. Dropout làm ‖W‖² lớn hơn, phạt L2 làm ‖W‖² nhỏ hơn, mà cả hai đều chống quá khớp. Vậy “trọng số nhỏ” có phải là lời
   giải thích duy nhất cho việc tổng quát hóa tốt?
6. Trong thí nghiệm đặc trưng ngẫu nhiên, vì sao lỗi kiểm định lại cao nhất đúng ở N = 1.000, và vì sao nó giảm lại khi
   N tiếp tục tăng?

<details><summary>Gợi ý đáp án</summary>

1. Mạng tự tin hơn ở cả những ảnh nó sai: mất mát trung bình trên ảnh sai tăng từ 2,85 lên 4,41, số ảnh sai mà tự tin
   ≥ 99% tăng từ 21 lên 132. Cross-entropy phạt nặng kiểu sai đó, còn độ chính xác thì không thấy.
2. Nếu chọn trên tập kiểm tra, con số báo cáo cũng là con số đã được tối ưu, nên nó lạc quan hơn thực tế. Tập kiểm tra
   chỉ đáng tin khi nó chưa ảnh hưởng tới bất kỳ lựa chọn nào.
3. $W \leftarrow (1 - 2\eta\lambda)W - \eta\nabla\mathcal{L}$, với $\eta = 0{,}1$ và $\lambda = 0{,}003$ là 0,9994 mỗi
   bước. Adam chia cả gradient lẫn phần $2\lambda W$ cho $\sqrt{\hat v}$, nên trọng số có gradient lớn bị co ít hơn
   (Bài 07); AdamW tách phép co ra khỏi phép chia đó.
4. Nhờ phép chia $1/(1-p)$ lúc huấn luyện, kỳ vọng của mỗi nơ-ron bằng giá trị của nó khi không có mặt nạ. Quên chia
   thì lớp sau quen với tín hiệu chỉ bằng 80% (với $p = 0{,}2$), và lúc suy luận nhận tín hiệu lớn gấp 1,25 lần.
5. Không. Dropout chống học thuộc bằng cách khác: mỗi nơ-ron phải hữu ích trong rất nhiều mạng con. Dịch ảnh cũng làm
   ‖W‖² lớn hơn mà lại giúp nhiều nhất. Zhang và cộng sự kết luận rằng chính quy hóa tường minh “không cần thiết, và
   tự nó cũng không đủ” để giải thích tổng quát hóa.
6. Ở N = 1.000 chỉ có một cách khớp đúng mọi ảnh, và vì $F$ gần suy biến nên cách đó cần $\lVert W\rVert$ rất lớn.
   Khi N lớn hơn, có vô số cách khớp, và nghiệm có $\lVert W\rVert$ nhỏ nhất ngày càng mượt.
</details>

## Tìm hiểu thêm
- G. Hinton et al. (2012), [Improving neural networks by preventing co-adaptation of feature
  detectors](https://arxiv.org/abs/1207.0580): bài báo đầu tiên về dropout.
- N. Srivastava et al. (2014), [Dropout: A Simple Way to Prevent Neural Networks from
  Overfitting](https://jmlr.org/papers/v15/srivastava14a.html): ở đó $p$ là xác suất *giữ lại*, thường 0,5–0,8 cho
  lớp ẩn.
- C. Zhang et al. (2017), [Understanding deep learning requires rethinking generalization](https://arxiv.org/abs/1611.03530):
  thí nghiệm nhãn ngẫu nhiên.
- M. Belkin et al. (2019), [Reconciling modern machine learning practice and the bias-variance
  trade-off](https://arxiv.org/abs/1812.11118): double descent với đặc trưng ngẫu nhiên.
- P. Nakkiran et al. (2019), [Deep Double Descent](https://arxiv.org/abs/1912.02292), và (2020)
  [Optimal Regularization Can Mitigate Double Descent](https://arxiv.org/abs/2003.01897).
- A. Karpathy (2019), [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/), mục
  “Regularize”: thêm dữ liệu thật trước, rồi tăng cường dữ liệu, dropout, weight decay, dừng sớm.
- [Dive into Deep Learning, mục 5.6 · Dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html).
- [MLU-Explain · Double Descent](https://mlu-explain.github.io/double-descent/) (tương tác).
- A. Chowdhery et al. (2022), [PaLM](https://arxiv.org/abs/2204.02311), mục 5; A. Radford et al. (2018),
  [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf).

**Bài tiếp theo:** [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/):
khi mạng sâu hàng chục lớp, tín hiệu và gradient tắt dần hoặc nổ tung, và khởi tạo đúng cách giữ chúng ổn định.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")